# llm-assist showcase (end-to-end)

This notebook demonstrates the full project idea from the repository:

- **EDA** on the bundled golden dataset (and optionally a labeled CSV)
- **Offline evaluation** (golden set)
- **Deep learning alternate path**: optional **BERT/RoBERTa** fine-tuning for category
- **Run the FastAPI service** via **uvicorn subprocess** and call the endpoints

Primary references:
- `README.md`
- `docs/IMPLEMENTATION_REPORT.md` (especially section **6**)
- `docs/METHODOLOGY_EDA_AND_DL.md`


## 0) Setup

This notebook uses subprocess calls to run scripts and start the API.

Install dependencies in your venv:

```bash
pip install -e ".[dev]"
pip install -e ".[eda]"
pip install -e ".[transformer]"
```

Notes:
- Outputs go under `artifacts/` (gitignored).
- The transformer training section can take time depending on hardware.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import requests

ROOT = Path.cwd().resolve()
assert (ROOT / "app").is_dir(), f"Run from repo root, got {ROOT}"

print("Repo root:", ROOT)
print("Python:", sys.version)
print("Executable:", sys.executable)


## 1) Project idea (what we built)

From `README.md`: a production-grade FastAPI service that uses an **LLM** to triage tickets, evaluate agent-response quality, and summarize threads, with optional RAG and offline eval.

For the **deep learning narrative** (implemented vs literature/future work), see `docs/IMPLEMENTATION_REPORT.md` section **6**.


In [ ]:
print((ROOT / "README.md").read_text(encoding="utf-8").splitlines()[0])
print()
print("Docs:")
print("-", ROOT / "docs" / "IMPLEMENTATION_REPORT.md")
print("-", ROOT / "docs" / "METHODOLOGY_EDA_AND_DL.md")


## 2) EDA (exploratory plots)

We generate plots from the bundled golden set (`data/golden/eval_set.jsonl`).

Outputs are written to `artifacts/eda/`.

In [ ]:
eda_out = ROOT / "artifacts" / "eda"

# Run EDA script (requires: pip install -e ".[eda]")
subprocess.check_call(
    [sys.executable, "scripts/run_eda.py"],
    cwd=str(ROOT),
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
)

print("EDA output dir:", eda_out)
print("Files:")
for p in sorted(eda_out.glob("*.png")):
    print("-", p.name)


In [ ]:
# Display figures inline
from IPython.display import Image, display

for p in sorted((ROOT / "artifacts" / "eda").glob("*.png")):
    display(Image(filename=str(p)))


## 3) Offline evaluation (golden set)

Runs the reproducible benchmark on `data/golden/eval_set.jsonl` and prints the markdown summary from `artifacts/eval/summary.md`.

In [ ]:
subprocess.check_call(
    [
        sys.executable,
        "scripts/run_offline_eval.py",
        "--data",
        "data/golden/eval_set.jsonl",
    ],
    cwd=str(ROOT),
    env={
        **os.environ,
        # settings validation in CI expects a long enough secret
        "APP_SECRET_KEY": os.environ.get("APP_SECRET_KEY", "test-secret-key-minimum-16chars"),
        "QUALITY_POLICY_CONTEXT_TOP_K": os.environ.get("QUALITY_POLICY_CONTEXT_TOP_K", "0"),
        "TRIAGE_POLICY_CONTEXT_TOP_K": os.environ.get("TRIAGE_POLICY_CONTEXT_TOP_K", "0"),
        "PYTHONUNBUFFERED": "1",
    },
)

summary_path = ROOT / "artifacts" / "eval" / "summary.md"
print(summary_path.read_text(encoding="utf-8"))


## 4) Deep learning demo (alternate method): fine-tune RoBERTa/BERT for category

This section runs a **real Hugging Face `Trainer`** fine-tune for the single task:

- input: CSV with columns `text,category`
- output: a checkpoint directory under `artifacts/` with `config.json`, tokenizer files, and weights

If you do not have a real labeled CSV downloaded yet, we generate a small demo CSV in `artifacts/demo_tickets.csv` so the training loop is still reproducible.

Install requirement:

```bash
pip install -e ".[transformer]"
```


In [ ]:
demo_csv = ROOT / "artifacts" / "demo_tickets.csv"

def write_demo_csv(path: Path) -> None:
    # 5 categories matching app.models.domain.Category
    rows = [
        ("charged twice for subscription", "billing"),
        ("refund not received for invoice", "billing"),
        ("cannot log in password reset fails", "authentication"),
        ("2FA code not working login blocked", "authentication"),
        ("export button crashes chrome", "technical_bug"),
        ("app freezes on settings screen", "technical_bug"),
        ("please add dark mode", "feature_request"),
        ("can you add SSO support", "feature_request"),
        ("what are your support hours", "general_inquiry"),
        ("where can I find API docs", "general_inquiry"),
    ]

    path.parent.mkdir(parents=True, exist_ok=True)
    lines = ["text,category\n"] + [f"{t},{c}\n" for t, c in rows]
    path.write_text("".join(lines), encoding="utf-8")

if not demo_csv.is_file():
    write_demo_csv(demo_csv)

print("Training CSV:", demo_csv)
print(demo_csv.read_text(encoding="utf-8"))


In [ ]:
model_dir = ROOT / "artifacts" / "triage_roberta_demo"

# Keep this short for a classroom demo
cmd = [
    sys.executable,
    "scripts/train_triage_transformer.py",
    "--data",
    str(demo_csv),
    "--out",
    str(model_dir),
    "--model",
    "roberta-base",
    "--epochs",
    "1",
    "--batch-size",
    "4",
]

print("Running:", " ".join(cmd))
subprocess.check_call(cmd, cwd=str(ROOT), env={**os.environ, "PYTHONUNBUFFERED": "1"})

metrics_path = model_dir / "train_metrics.json"
print("Checkpoint dir:", model_dir)
print("Metrics path:", metrics_path)
print(metrics_path.read_text(encoding="utf-8"))


To use this optional encoder inside the service, set:

- `TRIAGE_TRANSFORMER_ENABLED=true`
- `TRIAGE_TRANSFORMER_MODEL_DIR=artifacts/triage_roberta_demo`

This model is used only as a **category hint** prepended to the LLM prompt; the LLM remains responsible for returning the full structured triage JSON.


## 5) Run the API (uvicorn subprocess) and call endpoints

We start uvicorn in the background, poll `/api/v1/health`, then call:

- `/api/v1/triage`
- `/api/v1/quality`
- `/api/v1/pipeline`
- `/api/v1/summarize`
- `/api/v1/rag/context`

Finally, we shut the server down.


In [ ]:
API_BASE = "http://127.0.0.1:8000"

# Start uvicorn as a subprocess
env = {
    **os.environ,
    "APP_SECRET_KEY": os.environ.get("APP_SECRET_KEY", "test-secret-key-minimum-16chars"),
    "QUALITY_POLICY_CONTEXT_TOP_K": os.environ.get("QUALITY_POLICY_CONTEXT_TOP_K", "0"),
    "TRIAGE_POLICY_CONTEXT_TOP_K": os.environ.get("TRIAGE_POLICY_CONTEXT_TOP_K", "0"),
    # Enable the optional encoder hint we just trained (safe even if deps missing; service will skip)
    "TRIAGE_TRANSFORMER_ENABLED": "true",
    "TRIAGE_TRANSFORMER_MODEL_DIR": str(model_dir),
    "PYTHONUNBUFFERED": "1",
}

uvicorn_cmd = [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "8000"]
print("Starting:", " ".join(uvicorn_cmd))
proc = subprocess.Popen(
    uvicorn_cmd,
    cwd=str(ROOT),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# Wait for /health
deadline = time.time() + 30
last_err = None
while time.time() < deadline:
    try:
        r = requests.get(f"{API_BASE}/api/v1/health", timeout=1.5)
        if r.status_code == 200:
            print("API ready:", r.json())
            break
    except Exception as e:
        last_err = e
    time.sleep(0.5)
else:
    raise RuntimeError(f"API did not become ready. Last error: {last_err}")


In [ ]:
def post_json(path: str, payload: dict) -> dict:
    r = requests.post(f"{API_BASE}{path}", json=payload, timeout=30)
    r.raise_for_status()
    return r.json()

sample_ticket = "I was charged twice this month and need a refund immediately."

triage = post_json("/api/v1/triage", {"ticket_text": sample_ticket, "include_policy_context": False})
print("/triage:\n", json.dumps(triage, indent=2))

quality = post_json(
    "/api/v1/quality",
    {
        "ticket_text": sample_ticket,
        "agent_response": "Sorry about that. I opened a billing case and will refund within 24 hours.",
        "include_policy_context": False,
    },
)
print("\n/quality:\n", json.dumps(quality, indent=2))

pipeline = post_json(
    "/api/v1/pipeline",
    {
        "ticket_text": sample_ticket,
        "agent_response": "Sorry about that. I opened a billing case and will refund within 24 hours.",
    },
)
print("\n/pipeline:\n", json.dumps(pipeline, indent=2))

summ = post_json(
    "/api/v1/summarize",
    {
        "turns": [
            {"role": "customer", "content": "The app freezes when I open settings."},
            {
                "role": "agent",
                "content": "Please clear cache and update to 2.4. I will follow up tomorrow.",
            },
        ]
    },
)
print("\n/summarize:\n", json.dumps(summ, indent=2))

rag = post_json("/api/v1/rag/context", {"query": "refund timeline", "top_k": 2})
print("\n/rag/context:\n", json.dumps(rag, indent=2))


In [ ]:
# Shutdown uvicorn
try:
    proc.terminate()
    proc.wait(timeout=10)
except Exception:
    proc.kill()

# Print last ~200 lines of server output for debugging
if proc.stdout is not None:
    out = proc.stdout.read()
    lines = out.splitlines()[-200:]
    print("\n".join(lines))

print("Server stopped.")
